# Structured Outputs con Microsoft Agent Framework

## Descripción

En este ejercicio, aprenderás a crear agentes que producen **salidas estructuradas** en forma de objetos JSON que se ajustan a un esquema específico. Esto es fundamental cuando necesitas que la IA genere datos en un formato predecible y procesable.

### ¿Qué son Structured Outputs?

Las salidas estructuradas garantizan que las respuestas del agente sigan un formato JSON específico definido por un esquema. En lugar de obtener texto libre, obtienes datos estructurados que tu aplicación puede procesar de manera confiable.

### Casos de uso:
- 📦 Extracción de información de productos
- 📄 Análisis de documentos con formato consistente
- 🎫 Procesamiento de tickets y formularios
- 📊 Generación de reportes estructurados
- 🏷️ Clasificación y etiquetado de datos

### Cargar librerías

Importamos las bibliotecas necesarias. Usaremos **Pydantic** para definir los esquemas de datos.

In [ ]:
import os
import asyncio
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient

# GitHub Models client configuration
MODEL_NAME = os.getenv("GITHUB_MODEL", "openai/gpt-4o")

print("✓ Bibliotecas cargadas exitosamente")
print(f"✓ Modelo configurado: {MODEL_NAME}")

### Definir esquemas de datos con Pydantic

Usamos **Pydantic Models** para definir la estructura exacta que queremos que el agente genere. Pydantic proporciona validación de tipos y serialización automática.

#### Ejemplo 1: Información de Producto

In [ ]:
class ProductSpecifications(BaseModel):
    """Especificaciones técnicas del producto"""
    weight: Optional[str] = Field(None, description="Peso del producto")
    dimensions: Optional[str] = Field(None, description="Dimensiones del producto")
    material: Optional[str] = Field(None, description="Material principal")
    color: Optional[str] = Field(None, description="Color del producto")
    power: Optional[str] = Field(None, description="Especificaciones de energía")


class ProductInfo(BaseModel):
    """Información estructurada de un producto"""
    name: str = Field(description="Nombre del producto")
    category: str = Field(description="Categoría del producto")
    brand: str = Field(description="Marca del producto")
    price: float = Field(description="Precio en euros")
    description: str = Field(description="Descripción breve del producto")
    features: List[str] = Field(description="Lista de características principales")
    specifications: ProductSpecifications = Field(description="Especificaciones técnicas")
    in_stock: bool = Field(description="Disponibilidad en inventario")
    rating: Optional[float] = Field(None, description="Calificación promedio (0-5)")


print("✓ Esquema ProductInfo definido")
print("\nCampos del esquema:")
for field_name, field_info in ProductInfo.model_fields.items():
    print(f"  - {field_name}: {field_info.annotation.__name__ if hasattr(field_info.annotation, '__name__') else field_info.annotation}")

### Crear agente con salida estructurada

Ahora creamos un agente que extraerá información de productos de texto no estructurado y la convertirá en nuestro formato JSON definido.

In [ ]:
async def extract_product_info(product_description: str):
    """
    Extrae información estructurada de una descripción de producto en texto libre.
    
    Args:
        product_description: Descripción del producto en texto libre
        
    Returns:
        ProductInfo: Objeto con la información estructurada
    """
    
    print("\n" + "="*70)
    print("📦 EXTRACTOR DE INFORMACIÓN DE PRODUCTOS")
    print("="*70 + "\n")
    
    print("📝 Descripción del producto (texto no estructurado):")
    print("-" * 70)
    print(product_description)
    print("-" * 70 + "\n")
    
    # Configurar cliente
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    # Crear agente con formato de respuesta estructurado
    async with ChatAgent(
        chat_client=client,
        name="product_extractor",
        instructions="""Eres un experto en análisis de información de productos.
        
        Tu tarea es extraer y estructurar información de descripciones de productos.
        
        IMPORTANTE:
        - Extrae TODA la información disponible en la descripción
        - Si un campo no está disponible, usa null para campos opcionales
        - Normaliza los precios a formato numérico (sin símbolos de moneda)
        - Las características deben ser puntos claros y concisos
        - Sé preciso con las especificaciones técnicas
        
        Responde SOLO con el JSON estructurado, sin texto adicional.""",
        response_format=ProductInfo  # ⭐ Aquí especificamos el formato de salida
    ) as agent:
        
        print("🔄 Procesando con agente...\n")
        
        response = await agent.run([product_description])
        
        # La respuesta ya está en formato estructurado
        product_info = ProductInfo.model_validate_json(response.content)
        
        print("✅ Información extraída exitosamente\n")
        print("="*70)
        print("📊 RESULTADO ESTRUCTURADO (JSON)")
        print("="*70 + "\n")
        
        # Mostrar como JSON formateado
        print(json.dumps(product_info.model_dump(), indent=2, ensure_ascii=False))
        
        return product_info


# Ejemplo 1: Laptop
laptop_description = """Presentamos la nueva MacBook Pro 16 pulgadas de Apple, 
la laptop definitiva para profesionales. Con el revolucionario chip M3 Pro, 
16GB de RAM y 512GB de almacenamiento SSD. La pantalla Liquid Retina XDR 
ofrece un brillo increíble de 1000 nits. Incluye tres puertos Thunderbolt 4, 
ranura SDXC, y conector MagSafe 3. Batería de hasta 22 horas. Acabado en 
aluminio color gris espacial. Peso: 2.15 kg. Precio: 2899.99 EUR. 
Disponible en stock. Calificación de clientes: 4.8/5 estrellas."""

laptop_info = await extract_product_info(laptop_description)

print("\n" + "="*70)
print("✓ Extracción completada")
print("="*70)

### Ejemplo 2: Múltiples productos

Podemos definir esquemas más complejos para extraer múltiples productos a la vez.

In [ ]:
class ProductCatalog(BaseModel):
    """Catálogo con múltiples productos"""
    store_name: str = Field(description="Nombre de la tienda")
    category: str = Field(description="Categoría de los productos")
    products: List[ProductInfo] = Field(description="Lista de productos")
    total_products: int = Field(description="Número total de productos")


async def extract_product_catalog(catalog_text: str):
    """
    Extrae múltiples productos de un texto de catálogo.
    """
    
    print("\n" + "="*70)
    print("🏪 EXTRACTOR DE CATÁLOGO DE PRODUCTOS")
    print("="*70 + "\n")
    
    print("📝 Texto del catálogo:")
    print("-" * 70)
    print(catalog_text)
    print("-" * 70 + "\n")
    
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    async with ChatAgent(
        chat_client=client,
        name="catalog_extractor",
        instructions="""Eres un experto en análisis de catálogos de productos.
        
        Extrae TODOS los productos mencionados en el texto y organízalos en un catálogo estructurado.
        Identifica el nombre de la tienda y la categoría general.
        Cuenta correctamente el número total de productos.
        
        Responde SOLO con el JSON estructurado.""",
        response_format=ProductCatalog
    ) as agent:
        
        print("🔄 Procesando catálogo...\n")
        
        response = await agent.run([catalog_text])
        catalog = ProductCatalog.model_validate_json(response.content)
        
        print("✅ Catálogo extraído exitosamente\n")
        print("="*70)
        print("📊 CATÁLOGO ESTRUCTURADO")
        print("="*70 + "\n")
        
        print(f"🏪 Tienda: {catalog.store_name}")
        print(f"📂 Categoría: {catalog.category}")
        print(f"📦 Total de productos: {catalog.total_products}")
        print("\n" + "-" * 70 + "\n")
        
        for i, product in enumerate(catalog.products, 1):
            print(f"Producto {i}: {product.name}")
            print(f"  Marca: {product.brand}")
            print(f"  Precio: {product.price} EUR")
            print(f"  En stock: {'Sí' if product.in_stock else 'No'}")
            if product.rating:
                print(f"  Rating: {product.rating}/5 ⭐")
            print()
        
        print("\nJSON completo:")
        print(json.dumps(catalog.model_dump(), indent=2, ensure_ascii=False))
        
        return catalog


# Ejemplo de catálogo con múltiples productos
catalog_text = """Bienvenido a TechStore - Tu tienda de electrónica de confianza

Categoría: Smartphones

1. iPhone 15 Pro Max de Apple - El smartphone más avanzado del mercado. 
   Chip A17 Pro, cámara de 48MP, pantalla Super Retina XDR de 6.7 pulgadas.
   Titanio natural, 256GB de almacenamiento. Precio: 1349.99 EUR. 
   En stock. Rating: 4.9/5.

2. Samsung Galaxy S24 Ultra - Potencia y elegancia coreana.
   Snapdragon 8 Gen 3, 12GB RAM, 512GB almacenamiento, cámara 200MP.
   S Pen integrado, pantalla Dynamic AMOLED 2X de 6.8 pulgadas.
   Color negro titanio. Precio: 1299.99 EUR. En stock. Rating: 4.7/5.

3. Google Pixel 9 Pro - Inteligencia artificial en tu bolsillo.
   Tensor G4, 16GB RAM, cámara con IA avanzada, pantalla OLED de 6.3 pulgadas.
   Acabado mate en color gris tormenta, 256GB. Precio: 1099.99 EUR.
   Disponible para pedido anticipado (no en stock). Rating: 4.6/5.
"""

catalog = await extract_product_catalog(catalog_text)

print("\n" + "="*70)
print("✓ Procesamiento de catálogo completado")
print("="*70)

### Ejemplo 3: Análisis de reseñas con sentimiento

Otro caso de uso: extraer información estructurada de reseñas de clientes.

In [ ]:
from enum import Enum
from datetime import date


class Sentiment(str, Enum):
    """Sentimiento de la reseña"""
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"


class ReviewAnalysis(BaseModel):
    """Análisis estructurado de una reseña"""
    product_name: str = Field(description="Nombre del producto reseñado")
    reviewer_name: str = Field(description="Nombre del revisor")
    rating: int = Field(description="Calificación numérica (1-5)", ge=1, le=5)
    sentiment: Sentiment = Field(description="Sentimiento general")
    pros: List[str] = Field(description="Aspectos positivos mencionados")
    cons: List[str] = Field(description="Aspectos negativos mencionados")
    summary: str = Field(description="Resumen de la reseña en una oración")
    would_recommend: bool = Field(description="¿Recomendaría el producto?")
    verified_purchase: bool = Field(description="¿Es una compra verificada?")


async def analyze_review(review_text: str):
    """
    Analiza una reseña de producto y extrae información estructurada.
    """
    
    print("\n" + "="*70)
    print("⭐ ANALIZADOR DE RESEÑAS DE PRODUCTOS")
    print("="*70 + "\n")
    
    print("📝 Reseña original:")
    print("-" * 70)
    print(review_text)
    print("-" * 70 + "\n")
    
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    async with ChatAgent(
        chat_client=client,
        name="review_analyzer",
        instructions="""Eres un experto en análisis de sentimientos y reseñas de productos.
        
        Analiza la reseña y extrae:
        - Información del producto y revisor
        - Calificación y sentimiento
        - Pros y cons mencionados
        - Si recomienda el producto
        - Si es una compra verificada
        
        El sentimiento debe ser:
        - POSITIVE: rating 4-5 y comentarios mayormente positivos
        - NEUTRAL: rating 3 o mixto de pros/cons
        - NEGATIVE: rating 1-2 y comentarios mayormente negativos
        
        Responde SOLO con el JSON estructurado.""",
        response_format=ReviewAnalysis
    ) as agent:
        
        print("🔄 Analizando reseña...\n")
        
        response = await agent.run([review_text])
        analysis = ReviewAnalysis.model_validate_json(response.content)
        
        print("✅ Análisis completado\n")
        print("="*70)
        print("📊 ANÁLISIS ESTRUCTURADO")
        print("="*70 + "\n")
        
        sentiment_emoji = {"positive": "😊", "neutral": "😐", "negative": "😞"}
        
        print(f"📦 Producto: {analysis.product_name}")
        print(f"👤 Revisor: {analysis.reviewer_name}")
        print(f"⭐ Calificación: {analysis.rating}/5")
        print(f"{sentiment_emoji[analysis.sentiment]} Sentimiento: {analysis.sentiment.upper()}")
        print(f"✅ Compra verificada: {'Sí' if analysis.verified_purchase else 'No'}")
        print(f"👍 Recomendaría: {'Sí' if analysis.would_recommend else 'No'}")
        print(f"\n📝 Resumen: {analysis.summary}")
        
        print("\n✅ Pros:")
        for pro in analysis.pros:
            print(f"   + {pro}")
        
        if analysis.cons:
            print("\n❌ Cons:")
            for con in analysis.cons:
                print(f"   - {con}")
        
        print("\n" + "-" * 70)
        print("\nJSON completo:")
        print(json.dumps(analysis.model_dump(), indent=2, ensure_ascii=False))
        
        return analysis


# Ejemplo de reseña
review_text = """Reseña de: Sony WH-1000XM5
Por: María González
Calificación: 5 estrellas
Compra verificada

¡Estos auriculares son increíbles! La cancelación de ruido es la mejor que he probado,
perfecta para trabajar en cafeterías ruidosas. La calidad de sonido es excepcional,
con graves profundos y agudos cristalinos. La batería dura fácilmente 30 horas.
Son muy cómodos incluso después de horas de uso.

El único pequeño inconveniente es que el estuche es un poco grande para viajar,
y el precio es alto, pero totalmente vale la pena. Sin duda los recomendaría
a cualquiera que busque auriculares premium.
"""

analysis = await analyze_review(review_text)

print("\n" + "="*70)
print("✓ Análisis de reseña completado")
print("="*70)

## Resumen

### Conceptos clave aprendidos:

1. **Structured Outputs**: Garantizar que los agentes produzcan JSON con esquema definido

2. **Pydantic Models**: Definir esquemas de datos con:
   - Tipos de datos (str, int, float, bool, List, Optional)
   - Descripciones de campos
   - Validaciones (ge, le para rangos)
   - Modelos anidados (ProductSpecifications dentro de ProductInfo)
   - Enums para valores restringidos (Sentiment)

3. **Configuración de agentes**:
   ```python
   ChatAgent(
       response_format=YourPydanticModel  # ⭐ Clave para structured outputs
   )
   ```

4. **Casos de uso prácticos**:
   - 📦 Extracción de datos de productos
   - 🏪 Análisis de catálogos completos
   - ⭐ Análisis de reseñas con sentimiento
   - 📄 Procesamiento de documentos
   - 🎫 Extracción de datos de formularios

### Beneficios de Structured Outputs:
- ✅ **Confiabilidad**: Formato predecible garantizado
- ✅ **Validación automática**: Pydantic valida tipos y restricciones
- ✅ **Integración fácil**: JSON directo a objetos Python
- ✅ **Tipado fuerte**: Autocompletado y detección de errores
- ✅ **Procesamiento en batch**: Extraer múltiples entidades a la vez
- ✅ **APIs consistentes**: Respuestas uniformes para aplicaciones

### Mejores prácticas:
1. 📝 Proporciona descripciones claras en los campos
2. 🎯 Usa tipos específicos (Enum) cuando hay valores limitados
3. ✨ Marca campos como Optional cuando corresponda
4. 🔍 Valida las restricciones (rangos, longitudes)
5. 🏗️ Usa modelos anidados para estructuras complejas
6. 📋 Instruye al agente sobre el formato esperado

### Diferencia con respuestas de texto libre:
| Aspecto | Texto libre | Structured Output |
|---------|-------------|------------------|
| Formato | Inconsistente | Siempre JSON válido |
| Parsing | Regex/NLP complejo | Directo a objetos |
| Validación | Manual | Automática |
| Tipado | Strings | Tipos nativos |
| Confiabilidad | Variable | Garantizada |